In [1]:
import os
import sys
import torch
from pathlib import Path

if "__file__" in globals():
    project_root = Path(__file__).resolve().parent.parent
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if Path.cwd() != project_root:
    os.chdir(project_root)

from core.config import load_config, print_config
from core.data.loader import (
    setup_database, load_and_group_data_from_db, 
    calculate_normalization_stats, create_sampler_from_config, print_data_summary
)
from core.data.dataset import create_dataloaders
from core.models.factory import create_task_model_from_config, print_model_info
from core.training.trainer import setup_training
from core.data.types import HitObjectVector, BeatmapMetadata, VECTOR_DIM, METADATA_DIM

print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Working directory: {os.getcwd()}")

print(f"Vector fields: {HitObjectVector.get_field_names()}")
print(f"Metadata fields: {BeatmapMetadata.get_field_names()}")

PyTorch version: 2.8.0+cu126
Using device: cuda
Working directory: /home/jessiez/osu_corpora
Vector fields: ['x_diff', 'y_diff', 'object_type', 'is_new_combo', 'slider_curve_type', 'slider_num_anchors', 'slider_pixel_length', 'time_diff_bin', 'duration_bin', 'kiai_time']
Metadata fields: ['ar', 'od', 'cs', 'difficulty_rating', 'bpm']


In [2]:
CONFIG_NAME = "bert" 

config = load_config(CONFIG_NAME, config_dir="configs")
print_config(config, f"Loaded Configuration: {CONFIG_NAME}")


--- Loaded Configuration: bert ---
data:
  db_path: ./data/beatmaps_test.db
  max_seq_len: 1023
  val_split: 0.1
  chunk_size: 1000
training:
  batch_size: 8
  num_epochs: 5
  learning_rate: 0.0005
  weight_decay: 0.05
  warmup_ratio: 0.05
  min_lr: 1.0e-06
  use_amp: true
  checkpoint_dir: ./checkpoints/bert
  grad_clip_norm: 1.0
  use_class_weights: true
  class_weighting_method: inverse_frequency
  loss_weights:
    continuous: 1.0
    categorical: 0.5
  sampling:
    method: temperature
    kde_bandwidth: 0.3
    kde_bins: 200
    temperature: 2.0
    difficulty_index: 3
    expand_for_augmentation: true
model:
  dropout: 0.1
  type: bert
  d_model: 512
  n_heads: 8
  n_layers: 6
  dim_feedforward_mult: 4
components:
  use_rope: true
  use_flash_attention: true
  compile_model: true
mlm:
  masking_ratio: 0.25

----------------------------------


In [3]:
colab_url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
db_path = setup_database(config['data']['db_path'], colab_url)

print(f"Using database: {db_path}")

all_beatmaps_data = load_and_group_data_from_db(
    db_path, 
    chunk_size=config['data'].get('chunk_size', 1000),
    max_seq_len=config['data']['max_seq_len']
)

print_data_summary(all_beatmaps_data)

Using database: ./data/beatmaps_test.db
Connecting to database...
Fetching valid beatmap IDs and all metadata...
Found 2943 beatmaps with complete metadata.
Processing Chunks: 100%|██████████| 3/3 [00:05<00:00,  1.93s/it]
Finished processing all data.

--- Data Summary ---
Total beatmaps: 2943
Vector dimension: 10
Metadata dimension: 5
Sequence length - Min: 91, Max: 1023, Avg: 697.9
--------------------


In [5]:
from torch.utils.data import random_split

val_size = int(len(all_beatmaps_data) * config['data']['val_split'])
train_size = len(all_beatmaps_data) - val_size
train_data, val_data = random_split(all_beatmaps_data, [train_size, val_size])

print(f"Data split: {len(train_data)} training, {len(val_data)} validation")

train_data_list = [train_data.dataset[i] for i in train_data.indices]
val_data_list = [val_data.dataset[i] for i in val_data.indices]

sampler = create_sampler_from_config(train_data_list, config)

normalizer = calculate_normalization_stats(
    train_data_list,
    include_augmentation=config['training']['sampling']['expand_for_augmentation']
)

# Note: New normalizer interface returns dictionaries instead of tuples
vector_stats = normalizer.get_vector_stats()
meta_stats = normalizer.get_metadata_stats()

print(f"Vector normalization stats for {len(vector_stats)} fields")
print(f"Metadata normalization stats for {len(meta_stats)} fields")

Data split: 2649 training, 294 validation
Creating temperature sampler with temperature=2.0...
Temperature sampling - Min weight: 0.5590, Max weight: 300.0866
Calculating normalization statistics...
Including data augmentation in normalization statistics...

                    NORMALIZATION STATISTICS

--- VECTOR STATISTICS:
----------------------------------------------------------------------
Field Name           Type         Param 1      Param 2      Description
----------------------------------------------------------------------
x_diff               mean/std     0.0000       119.8255     X-coordinate difference
y_diff               mean/std     0.0000       108.0182     Y-coordinate difference
object_type          categorical  N/A          N/A          Object type (categorical)
is_new_combo         categorical  N/A          N/A          New combo flag (categorical)
slider_curve_type    categorical  N/A          N/A          Slider curve type (categorical)
slider_num_anchors   lo

In [6]:
train_dataloader, val_dataloader = create_dataloaders(
    train_data_list,
    val_data_list,
    normalizer,
    config, device, sampler
)

print(f"Created dataloaders with batch size: {config['training']['batch_size']}")

sample_batch = next(iter(train_dataloader))
print(f"Sample batch shapes: vectors={sample_batch[0].shape}, mask={sample_batch[1].shape}, meta={sample_batch[2].shape}")

Created dataloaders with batch size: 8
Sample batch shapes: vectors=torch.Size([8, 1023, 10]), mask=torch.Size([8, 1023]), meta=torch.Size([8, 5])
Sample batch shapes: vectors=torch.Size([8, 1023, 10]), mask=torch.Size([8, 1023]), meta=torch.Size([8, 5])


In [7]:
model = create_task_model_from_config(config, device, task_type='mlm')

print_model_info(model, config)

print("\nRunning a test forward pass with mixed precision (autocast)...")
with torch.no_grad():
    with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
        sample_vectors, sample_mask, sample_metadata = sample_batch
        
        sample_vectors = sample_vectors.to(device)
        sample_mask = sample_mask.to(device)
        sample_metadata = sample_metadata.to(device)

        predictions, targets, _ = model(sample_vectors, sample_metadata, sample_mask)

print("\n✅ Model created and tested successfully!")

Compiling task model with torch.compile...

--- Model Information ---
Model Type: bert
Total Parameters: 25.21M
Model Dimension: 512
Number of Heads: 8
Number of Layers: 6
RoPE Enabled: True
Flash Attention: True
Model Compiled: True
-------------------------

Running a test forward pass with mixed precision (autocast)...

--- Model Information ---
Model Type: bert
Total Parameters: 25.21M
Model Dimension: 512
Number of Heads: 8
Number of Layers: 6
RoPE Enabled: True
Flash Attention: True
Model Compiled: True
-------------------------

Running a test forward pass with mixed precision (autocast)...


W0917 18:37:46.256000 503962 .venv/lib/python3.12/site-packages/torch/_inductor/utils.py:1436] [4/0_1] Not enough SMs to use max_autotune_gemm mode



✅ Model created and tested successfully!


In [8]:
trainer, checkpoint_manager = setup_training(
    model, train_dataloader, val_dataloader, config, device
)

start_epoch = 0
if checkpoint_manager.checkpoint_exists():
    try:
        start_epoch, metrics = checkpoint_manager.load_checkpoint(
            model, trainer.optimizer, trainer.scheduler, trainer.scaler, device=device
        )
        start_epoch += 1 
        print(f"Loaded checkpoint, resuming from epoch {start_epoch + 1}")
        print(f"Previous metrics: {metrics}")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")
        print("Starting training from scratch")

print(f"Training setup complete. Starting from epoch {start_epoch + 1}")
print(f"Total epochs: {config['training']['num_epochs']}")

Trainer initialized - AMP: True, Device: cuda
Training setup complete. Starting from epoch 1
Total epochs: 5


In [9]:

print("\n🚀 Starting training...")
print(f"Configuration: {CONFIG_NAME}")
print(f"Model: {config['model']['type']} with {config['model']['n_layers']} layers")
print(f"Components: RoPE={config['components'].get('use_rope', False)}")

if config.get('training', {}).get('sampling', {}).get('expand_for_augmentation', True):
    effective_train_size = len(train_data) * 4 if config.get('training', {}).get('sampling', {}).get('expand_for_augmentation', True) else len(train_data)
    print(f"Training samples: {len(train_data)} base maps -> {effective_train_size} with augmentation")
else:
    print(f"Training samples: {len(train_data)} base maps without augmentation")

metrics_tracker = trainer.train(start_epoch)

print("\n🎉 Training completed!")
print("Training history saved in metrics_tracker")


🚀 Starting training...
Configuration: bert
Model: bert with 6 layers
Components: RoPE=True
Training samples: 2649 base maps -> 10596 with augmentation

--- Starting Training ---
Epochs: 1 to 5
Batch Size: 8
Learning Rate: 0.0005
------------------------------------------------------------


Epoch 1 [Train]:   0%|          | 0/1325 [00:00<?, ?it/s]

Epoch 1/5 | Train Loss: 6.0651 | Val Loss: 5.2937 | LR: 4.70e-04 | Time: 140.02s
                      DETAILED VALIDATION REPORT                       
----------------------------------------------------------------------
 CONTINUOUS FEATURES:
  Feature                | MAE        | Mean (True)  | Std (True)  
  -----------------------+------------+--------------+-------------
  x_diff                 | 0.6046     | 0.0024       | 0.9932      
  y_diff                 | 0.5923     | -0.0028      | 0.9966      
  slider_num_anchors     | 0.5724     | -0.0040      | 0.9904      
  slider_pixel_length    | 0.5776     | 0.0017       | 1.0029      
----------------------------------------------------------------------
 CATEGORICAL FEATURES:
  Feature                | Accuracy   | Precision    | Recall      
  -----------------------+------------+--------------+-------------
  object_type            | 78.79%     | 0.5217       | 0.5010      
    └─ True Dist: 0:63.0%, 1:36.9%, 2:0.1%
  is_

Epoch 2 [Train]:   0%|          | 0/1325 [00:00<?, ?it/s]

Epoch 2/5 | Train Loss: 4.4203 | Val Loss: 4.6820 | LR: 3.51e-04 | Time: 121.75s
                      DETAILED VALIDATION REPORT                       
----------------------------------------------------------------------
 CONTINUOUS FEATURES:
  Feature                | MAE        | Mean (True)  | Std (True)  
  -----------------------+------------+--------------+-------------
  x_diff                 | 0.5857     | -0.0005      | 1.0012      
  y_diff                 | 0.5733     | 0.0096       | 0.9931      
  slider_num_anchors     | 0.5267     | -0.0031      | 0.9909      
  slider_pixel_length    | 0.5064     | 0.0024       | 1.0026      
----------------------------------------------------------------------
 CATEGORICAL FEATURES:
  Feature                | Accuracy   | Precision    | Recall      
  -----------------------+------------+--------------+-------------
  object_type            | 82.32%     | 0.6407       | 0.5422      
    └─ True Dist: 0:63.0%, 1:36.9%, 2:0.1%
  is_

...

Epoch 3/5 | Train Loss: 3.9669 | Val Loss: 4.3483 | LR: 1.89e-04 | Time: 131.22s
                      DETAILED VALIDATION REPORT                       
----------------------------------------------------------------------
 CONTINUOUS FEATURES:
  Feature                | MAE        | Mean (True)  | Std (True)  
  -----------------------+------------+--------------+-------------
  x_diff                 | 0.5698     | 0.0068       | 0.9952      
  y_diff                 | 0.5558     | -0.0001      | 0.9929      
  slider_num_anchors     | 0.4540     | -0.0095      | 0.9890      
  slider_pixel_length    | 0.4571     | -0.0044      | 1.0007      
----------------------------------------------------------------------
 CATEGORICAL FEATURES:
  Feature                | Accuracy   | Precision    | Recall      
  -----------------------+------------+--------------+-------------
  object_type            | 84.23%     | 0.6397       | 0.5701      
    └─ True Dist: 0:63.3%, 1:36.6%, 2:0.1%
  is_

Epoch 4 [Train]:   0%|          | 0/1325 [00:00<?, ?it/s]

Epoch 4/5 | Train Loss: 3.6510 | Val Loss: 4.0476 | LR: 5.36e-05 | Time: 127.55s
                      DETAILED VALIDATION REPORT                       
----------------------------------------------------------------------
 CONTINUOUS FEATURES:
  Feature                | MAE        | Mean (True)  | Std (True)  
  -----------------------+------------+--------------+-------------
  x_diff                 | 0.5579     | 0.0008       | 0.9976      
  y_diff                 | 0.5496     | -0.0002      | 0.9932      
  slider_num_anchors     | 0.4241     | -0.0081      | 0.9917      
  slider_pixel_length    | 0.4269     | -0.0037      | 1.0018      
----------------------------------------------------------------------
 CATEGORICAL FEATURES:
  Feature                | Accuracy   | Precision    | Recall      
  -----------------------+------------+--------------+-------------
  object_type            | 85.76%     | 0.6784       | 0.5727      
    └─ True Dist: 0:63.3%, 1:36.6%, 2:0.1%
  is_

Epoch 5 [Train]:   0%|          | 0/1325 [00:00<?, ?it/s]

Epoch 5/5 | Train Loss: 3.5436 | Val Loss: 4.0268 | LR: 1.00e-06 | Time: 120.86s
                      DETAILED VALIDATION REPORT                       
----------------------------------------------------------------------
 CONTINUOUS FEATURES:
  Feature                | MAE        | Mean (True)  | Std (True)  
  -----------------------+------------+--------------+-------------
  x_diff                 | 0.5574     | 0.0014       | 1.0046      
  y_diff                 | 0.5462     | -0.0017      | 0.9962      
  slider_num_anchors     | 0.4180     | -0.0046      | 0.9906      
  slider_pixel_length    | 0.4198     | 0.0001       | 1.0012      
----------------------------------------------------------------------
 CATEGORICAL FEATURES:
  Feature                | Accuracy   | Precision    | Recall      
  -----------------------+------------+--------------+-------------
  object_type            | 85.88%     | 0.7766       | 0.5923      
    └─ True Dist: 0:63.1%, 1:36.8%, 2:0.1%
  is_

In [ ]:
if checkpoint_manager.checkpoint_exists():
    try:
        epoch, metrics = checkpoint_manager.load_checkpoint(
            model, trainer.optimizer, trainer.scheduler, trainer.scaler, device=device
        )
        print(f"Loaded checkpoint from epoch {epoch + 1}")
        
        print("\nRunning validation...")
        val_metrics = trainer.validate_epoch(epoch)
        
        print(f"Validation Loss: {val_metrics['loss']:.4f}")
        
        if 'mae_per_feature' in val_metrics:
            print("\nValidation MAE per feature:")
            mae_list = val_metrics['mae_per_feature']
            vector_field_names = HitObjectVector.get_field_names()
            max_name_len = max(len(name) for name in vector_field_names)
            for i in range(0, len(vector_field_names), 3):
                line = "  "
                for j in range(3):
                    if i + j < len(vector_field_names):
                        name = vector_field_names[i+j]
                        mae = mae_list[i+j]
                        line += f"{name:<{max_name_len}}: {mae:.4f} | "
                print(line.strip().rstrip('|').strip())
                
    except Ex
    print("No checkpoint found for validation")